# QIR Inputs and Interactive Workflows

This notebook shows how to:
- Build from QIR LLVM IR, bitcode bytes, and files
- Use `InteractiveSimulator`, `InteractiveRuntime`, and `InteractiveFullStack`

These APIs are useful for low-level debugging and incremental experimentation.

In [ ]:
from pathlib import Path
import tempfile

from qir_qis import qir_ll_to_bc, get_entry_attributes

from selene_sim import Stim, Quest, SimpleRuntime
from selene_sim.build import build
from selene_sim.interactive import InteractiveSimulator, InteractiveRuntime, InteractiveFullStack

## 1) Build from QIR in multiple formats

The same QIR can be passed as `.ll` text, `.bc` bytes, or file paths.

In [ ]:
RESOURCE_DIR = Path("../tests/resources/qir").resolve()
ll_file = RESOURCE_DIR / "adaptive_cond_loop.ll"
ll_text = ll_file.read_text()
bc_bytes = qir_ll_to_bc(ll_text)

attrs = get_entry_attributes(bc_bytes)
n_qubits = int(attrs.get("required_num_qubits", "6"))

with tempfile.NamedTemporaryFile(suffix=".bc", delete=False) as f:
    bc_file = Path(f.name)
bc_file.write_bytes(bc_bytes)

programs = {
    "ll_file": ll_file,
    "ll_text": ll_text,
    "bc_bytes": bc_bytes,
    "bc_file": bc_file,
}

results = {}
for name, input_program in programs.items():
    qir_runner = build(input_program, "no_results")
    results[name] = list(qir_runner.run(Stim(random_seed=101), n_qubits=n_qubits))

print(results)

## 2) Interactive simulator

Step-by-step control of native operations and immediate measurement queries.

In [ ]:
from math import pi

sim = InteractiveSimulator(simulator=Quest(random_seed=1234), n_qubits=4)
sim.reset(0)
sim.rxy(0, pi / 2, -pi / 2)
sim.rz(0, pi)
bit = sim.measure(0)

print("measurement:", bit)
print("metrics:", sim.get_metrics())

## 3) Interactive runtime

Queue runtime operations and flush when forcing futures.

In [ ]:
runtime = InteractiveRuntime(runtime=SimpleRuntime(), n_qubits=4)
q = runtime.qalloc()
runtime.reset(q)
runtime.rxy(q, pi, 0)
future = runtime.measure(q)

ops_before = runtime.get_operations()
runtime.force_result(future)
ops_after = runtime.get_operations()

print("ops captured before force_result:", len(ops_before))
print("ops remaining after force_result:", len(ops_after))

## 4) Interactive full stack

Combine runtime + simulator in one object for end-to-end interactive sessions.

In [ ]:
stack = InteractiveFullStack(simulator=Quest(random_seed=55), n_qubits=4)
q0 = stack.qalloc()
q1 = stack.qalloc()

stack.rxy(q0, pi / 2, -pi / 2)
stack.rz(q0, pi)
stack.rzz(q0, q1, pi / 2)

m0 = stack.measure(q0)
m1 = stack.measure(q1)

print("measurements:", m0, m1)